# Single-Runner SAC Trader

A new approach: the agent **observes the whole race** (raw book + engineered features for every runner) but
**trades one runner per episode** (1st, 2nd, ... nth favourite). Each runner of each race is a separate episode.
It starts with $100, can back and lay any size at any ladder price every step, gets Betfair-like fills
(including cross-matching), and is **greened up without bias at the off**.

The code lives in `single_runner/` (see `single_runner/README.md`). This notebook only drives it.

In [ ]:
### CELL 1 - DRIVE + CODE ###
from google.colab import drive
drive.mount('/content/drive')

!git clone -q -b claude/pensive-bell-6dmlvb https://github.com/tristanhowells/AHR_RL.git /content/AHR_RL 2>/dev/null || git -C /content/AHR_RL pull -q
%cd /content/AHR_RL
!pip -q install gymnasium pyarrow

In [ ]:
### CELL 2 - CONFIG ###
DATA_DIR = '/content/drive/MyDrive/race_out'
OUT_DIR  = '/content/drive/MyDrive/Betfair_RL/SingleRunner_SAC_v1'

RANKS = None            # e.g. (1, 2, 3) to train only on the top-3 favourites; None = every runner
VAL_START = '2026-02-01'  # chronological split; None -> last 20% of dates
TOTAL_STEPS = 1_000_000
EVAL_EVERY = 50_000
MAX_EVAL_EPISODES = 2000

from single_runner import EnvConfig, SACConfig
env_cfg = EnvConfig(
    initial_balance=100.0,
    tick_range=10,          # price action spans +/- 10 ladder ticks around the touch
    min_stake=1.0,          # set to your Betfair minimum
    fill_optimism=1.0,      # < 1 = less optimistic passive fills
    cross_matching=True,
    greenup_mode='fair',    # 'fair' (unbiased) | 'cross' (pays spread, stress test)
    reward_scale=10.0,
    runner_ranks=RANKS,
)
sac_cfg = SACConfig(buffer_size=300_000, batch_size=256, lr=3e-4, gamma=0.99,
                    learning_starts=5_000, updates_per_step=1)

In [ ]:
### CELL 3 - DATA SPLIT + BASELINES ###
import numpy as np
from single_runner.train import list_race_files, temporal_split, evaluate, summarise
from single_runner import SingleRunnerTradingEnv, episode_specs

files = list_race_files(DATA_DIR)
train_files, val_files = temporal_split(files, val_frac=0.2, val_start=VAL_START)
print(f'{len(files)} races -> train {len(train_files)} / val {len(val_files)}')

# Baselines on a slice of validation episodes: do-nothing is exactly 0, random should lose
class _Const:
    def __init__(self, a): self.a = np.asarray(a, np.float32)
    def act(self, obs, deterministic=True): return self.a
class _Random:
    def act(self, obs, deterministic=True): return np.random.uniform(-1, 1, 6).astype(np.float32)

probe_env = SingleRunnerTradingEnv(val_files, env_cfg)
probe = episode_specs(probe_env, val_files[:20])
for name, pol in [('do-nothing', _Const(-np.ones(6))), ('random', _Random())]:
    s = summarise(evaluate(pol, probe_env, probe))
    print(f"{name:11s} mean pnl {s['mean_pnl']:+.2f}  win {s['win_rate']:.2f}  matched {s['mean_matched']:.0f}")

In [ ]:
### CELL 4 - TRAIN ###
from single_runner.train import train
agent = train(train_files, val_files, OUT_DIR, env_cfg, sac_cfg,
              total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY,
              max_eval_episodes=MAX_EVAL_EPISODES, seed=42)

In [ ]:
### CELL 5 - EVALUATE BEST CHECKPOINT ON ALL VALIDATION EPISODES ###
import pandas as pd
from dataclasses import replace
from single_runner import SACAgent

best, extra = SACAgent.load(f'{OUT_DIR}/best.pt')
print('best checkpoint step', extra.get('step'))
for mode in ['fair', 'cross']:   # 'cross' re-scores the same policy with a spread-paying green-up
    env = SingleRunnerTradingEnv(val_files, replace(env_cfg, greenup_mode=mode, random_start=False))
    rows = pd.DataFrame(evaluate(best, env, episode_specs(env, val_files)))
    print(f'\n=== green-up: {mode} ===  mean {rows.pnl.mean():+.3f}  total {rows.pnl.sum():+.1f}  '
          f'win {(rows.pnl > 0).mean():.2f}  trade {((rows.back_matched + rows.lay_matched) > 0).mean():.2f}')
    print(rows.groupby('target_rank').agg(n=('pnl', 'size'), mean_pnl=('pnl', 'mean'),
                                          win=('pnl', lambda x: (x > 0).mean()),
                                          matched=('back_matched', 'mean')).round(3))
rows.to_csv(f'{OUT_DIR}/val_episodes_best.csv', index=False)